# 06 – WAR model v0.4 (single season)

**Project 1** is a single-season WAR calculator and Baseball Savant-style player cards. WAR here answers one question: **how much did this player help his team win in this season?** It is a record of what happened, not a prediction. One-year wonders are allowed. Separating skill from luck across seasons is Project 2's job (the team simulator).

**What changes in v0.4:**
1. **Finishing counts fully.** Goals above expected were weighted 0.3 in v0.3. In a "what happened" number, a goal is a goal, so the weight goes to 1.0.
2. **Defense: three alternatives tested, current method kept.** None of them measured defense meaningfully better (section 4).
3. **WAR split into Offense, Defense, and Other** for the player card. The three add up to total WAR.
4. **New validation test.** Add up each team's WAR and check it against the team's actual goal differential. A good single-season WAR should line up closely.

Data: MoneyPuck.com season-level skater files, `data/skaters21-22.csv` through `data/skaters25-26.csv`.

## 1. Setup

In [ ]:
import sys
sys.path.append("../src")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from war_model import run_season, rate60, COMPONENTS, V03_PARAMS, V04_PARAMS

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)

DATA = Path("../data")
IMAGES = Path("../images")
SEASONS = {2021: "21-22", 2022: "22-23", 2023: "23-24", 2024: "24-25", 2025: "25-26"}
GOALS_PER_WIN = V04_PARAMS["goals_per_win"]

print("Finishing weight  v0.3:", V03_PARAMS["finish_weight"], "  v0.4:", V04_PARAMS["finish_weight"])

## 2. Run v0.3 and v0.4 on every season

In [ ]:
v03 = pd.concat([run_season(DATA / f"skaters{l}.csv", V03_PARAMS) for l in SEASONS.values()], ignore_index=True)
v04 = pd.concat([run_season(DATA / f"skaters{l}.csv", V04_PARAMS) for l in SEASONS.values()], ignore_index=True)

# Check: v0.3 still matches notebook 04 exactly
old = pd.read_csv(DATA / "war_v03.csv")[["playerId", "war"]]
check = old.merge(v03[v03["season"] == 2025][["playerId", "war"]], on="playerId", suffixes=("_nb04", "_now"))
print("v0.3 reproduced:", len(check), "players, largest difference", float((check["war_nb04"] - check["war_now"]).abs().max()))

## 3. Change 1: finishing counts fully
How the 2025-26 top 15 moves when goals above expected count fully.

In [ ]:
cur03 = v03[v03["season"] == 2025][["playerId", "war", "war_rank"]]
cur04 = v04[v04["season"] == 2025]
compare = cur04.merge(cur03, on="playerId", suffixes=("_v04", "_v03"))
compare["ev_goals_minus_xg"] = compare["ev_I_F_goals"] - compare["ev_I_F_xGoals"]

(compare.nsmallest(15, "war_rank_v04")
        [["name", "team", "position", "war_v03", "war_rank_v03", "war_v04", "war_rank_v04", "ev_goals_minus_xg"]]
        .round(2))

In [ ]:
rho = compare["war_v03"].rank().corr(compare["war_v04"].rank())
print(f"Rank correlation, v0.3 vs v0.4 (all skaters): {rho:.3f}")

movers = compare.assign(change=compare["war_rank_v03"] - compare["war_rank_v04"])
print("\nBiggest risers into the top 100:")
print(movers[movers["war_rank_v04"] <= 100].nlargest(5, "change")[["name", "war_rank_v03", "war_rank_v04", "ev_goals_minus_xg"]].round(1).to_string(index=False))
print("\nBiggest fallers out of the v0.3 top 100:")
print(movers[movers["war_rank_v03"] <= 100].nsmallest(5, "change")[["name", "war_rank_v03", "war_rank_v04", "ev_goals_minus_xg"]].round(1).to_string(index=False))

**Caufield is now #1.** That is the honest result of this definition: he scored about 15 more even-strength goals than expected, and in a "what happened" WAR those goals count. Whether it repeats is a Project 2 question.

## 4. Change 2: can the defense stat be improved?
The current defense stat compares the team's expected goals against with the player on the ice vs. on the bench.

**Correction to notebook 05:** notebook 05 tested defense against next season's *goal differential* and found nothing. That was the wrong test, because good defensive forwards tend to score less, which drags goal differential down. The right test for a defense stat is whether it predicts next season's **goals against**. On that test, it passes weakly.

Four versions, each tested for repeatability and for predicting next season's relative goals against (higher = better defense):
- **A. Current:** on-ice vs. bench expected goals against
- **B. Zone-start adjusted:** A, corrected for how often the player starts shifts in his own zone
- **C. Team-relative:** on-ice expected goals against vs. the team's average
- **D. Score/venue adjusted:** A, using MoneyPuck's expected goals adjusted for score, home ice, and rebound flurries

In [ ]:
EXTRA = ["I_F_oZoneShiftStarts", "I_F_dZoneShiftStarts", "OnIce_A_goals", "OnIce_A_flurryScoreVenueAdjustedxGoals"]
extra = []
for label in SEASONS.values():
    raw = pd.read_csv(DATA / f"skaters{label}.csv")
    ev = raw.loc[raw["situation"] == "5on5", ["playerId", "season"] + EXTRA]
    extra.append(ev.rename(columns={c: f"ev_{c}" for c in EXTRA}))
d = v04.merge(pd.concat(extra), on=["playerId", "season"], how="left")

team = d.groupby(["season", "team"])
team_ga = team["ev_OnIce_A_goals"].transform("sum") / 5          # five skaters on the ice at 5v5
team_xga = team["ev_OnIce_A_xGoals"].transform("sum") / 5
team_toi = team["ev_toi"].transform("sum") / 5

# Target: relative goals against (team GA/60 without him minus GA/60 with him; higher = better)
d["rel_ga_60"] = rate60(team_ga - d["ev_OnIce_A_goals"], d["ev_bench_toi"]) - rate60(d["ev_OnIce_A_goals"], d["ev_toi"])

d["A_current"] = d["ev_rel_xga_60"]
d["C_team_relative"] = rate60(team_xga, team_toi) - rate60(d["ev_OnIce_A_xGoals"], d["ev_toi"])
d["D_score_adjusted"] = rate60(d["ev_OffIce_A_xGoals"], d["ev_bench_toi"]) - rate60(d["ev_OnIce_A_flurryScoreVenueAdjustedxGoals"], d["ev_toi"])

q = d[d["ev_toi"] >= 500].copy()
starts = q["ev_I_F_oZoneShiftStarts"] + q["ev_I_F_dZoneShiftStarts"]
q["dz_share"] = np.where(starts > 0, q["ev_I_F_dZoneShiftStarts"] / starts, np.nan)
q["dz_share"] = q["dz_share"].fillna(q["dz_share"].mean())
q["B_zone_adjusted"] = np.nan
for pos in ["F", "D"]:
    m = q["pos_group"] == pos
    X = np.column_stack([np.ones(m.sum()), q.loc[m, "dz_share"]])
    coef = np.linalg.lstsq(X, q.loc[m, "A_current"], rcond=None)[0]
    q.loc[m, "B_zone_adjusted"] = q.loc[m, "A_current"] - coef[1] * (q.loc[m, "dz_share"] - q.loc[m, "dz_share"].mean())

pairs = pd.concat([q[q["season"] == s].merge(q[q["season"] == s + 1], on="playerId", suffixes=("_y1", "_y2"))
                   for s in sorted(q["season"].unique())[:-1]], ignore_index=True)

CANDIDATES = ["A_current", "B_zone_adjusted", "C_team_relative", "D_score_adjusted", "rel_ga_60"]
rows = []
for c in CANDIDATES:
    row = {"version": c}
    for pos, name in [("F", "Forwards"), ("D", "Defense")]:
        g = pairs[pairs["pos_group_y1"] == pos]
        row[f"{name}: repeatable"] = g[f"{c}_y1"].corr(g[f"{c}_y2"])
        row[f"{name}: predicts next GA"] = g[f"{c}_y1"].corr(g["rel_ga_60_y2"])
    rows.append(row)
pd.DataFrame(rows).set_index("version").round(2)

**Decision: keep A.** B and C are worse or equal. D is better by about 0.01 to 0.02, which is too small to be worth the extra complexity. The last row is a useful reference: actual goals against only repeat at about 0.18, so the current stat (based on expected goals) is *more* stable than the real thing it estimates.

The defense stat still can't separate a player from his linemates and opponents. Fixing that needs shift-level data (who was on the ice with whom). That is a later project.

## 5. Offense, Defense, Other
For the player card, WAR is split into three parts that add up to the total:
- **Offensive WAR:** even-strength offense + power play offense
- **Defensive WAR:** even-strength defense + penalty kill defense
- **Other WAR:** penalties drawn minus taken, and faceoffs

In [ ]:
cur = v04[v04["season"] == 2025].copy()
print("Split adds up:", np.allclose(cur["off_war"] + cur["def_war"] + cur["other_war"], cur["war"]))
cur.nsmallest(15, "war_rank")[["war_rank", "name", "team", "position", "off_war", "def_war", "other_war", "war"]].round(2)

## 6. Validation: does team WAR match team results?
Add up every player's WAR for each team, then compare it to the team's actual 5-on-5 goal differential. Team 5v5 goals = the sum of every skater's on-ice goals divided by 5. A correlation near 1 means WAR is distributing the team's real results to its players accurately.

This is only a partial check (it uses 5v5 goal differential, while WAR also covers the power play and penalty kill), but it is fair for comparing v0.3 to v0.4.

In [ ]:
def team_check(war_table):
    out = {}
    for season, label in SEASONS.items():
        raw = pd.read_csv(DATA / f"skaters{label}.csv")
        ev = raw.loc[raw["situation"] == "5on5", ["playerId", "OnIce_F_goals", "OnIce_A_goals"]]
        w = war_table[war_table["season"] == season].merge(ev, on="playerId")
        t = w.groupby("team").agg(war=("war", "sum"), gf=("OnIce_F_goals", "sum"), ga=("OnIce_A_goals", "sum"))
        t["gd_5v5"] = (t["gf"] - t["ga"]) / 5
        out[f"{season}-{str(season + 1)[-2:]}"] = t["war"].corr(t["gd_5v5"])
    return pd.Series(out)

team_fit = pd.DataFrame({"v0.3": team_check(v03), "v0.4": team_check(v04)})
team_fit.loc["average"] = team_fit.mean()
team_fit.round(3)

## 7. Chart: what drives the 2025-26 top 15 (v0.4)

In [ ]:
COLORS = {"ev_off_gar": "#2a78d6", "pp_off_gar": "#eb6834", "ev_def_gar": "#1baf7a",
          "pk_def_gar": "#eda100", "pen_gar": "#e87ba4", "fo_gar": "#008300"}
LABELS = {"ev_off_gar": "EV offense", "pp_off_gar": "PP offense", "ev_def_gar": "EV defense",
          "pk_def_gar": "PK defense", "pen_gar": "Penalties", "fo_gar": "Faceoffs"}
SURFACE, INK, MUTED = "#fcfcfb", "#0b0b0b", "#52514e"

def component_chart(df, title, path):
    top = df.sort_values("war", ascending=False).head(15).iloc[::-1]
    comp = top[list(COLORS)] / GOALS_PER_WIN
    fig, ax = plt.subplots(figsize=(9, 7), facecolor=SURFACE)
    ax.set_facecolor(SURFACE)
    pos_left, neg_left = np.zeros(len(top)), np.zeros(len(top))
    for col, color in COLORS.items():
        vals = comp[col].to_numpy()
        ax.barh(top["name"], vals, left=np.where(vals >= 0, pos_left, neg_left), color=color,
                height=0.62, edgecolor=SURFACE, linewidth=2, label=LABELS[col])
        pos_left += np.where(vals >= 0, vals, 0)
        neg_left += np.where(vals < 0, vals, 0)
    for y, (total, end) in enumerate(zip(top["war"], pos_left)):
        ax.text(end + 0.05, y, f"{total:.1f}", va="center", fontsize=9, color=INK)
    ax.axvline(0, color=MUTED, linewidth=1)
    ax.set_xlabel("Wins above replacement", color=MUTED)
    ax.set_title(title, loc="left", color=INK, fontsize=12)
    ax.tick_params(colors=MUTED, labelsize=9)
    ax.grid(axis="x", color="#e6e5e1", linewidth=0.8)
    ax.set_axisbelow(True)
    for spine in ax.spines.values():
        spine.set_visible(False)
    ax.legend(loc="lower right", frameon=False, fontsize=9, labelcolor=MUTED)
    fig.text(0.01, 0.01, "Data: MoneyPuck.com", fontsize=8, color=MUTED)
    plt.tight_layout()
    fig.savefig(path, dpi=150, facecolor=SURFACE)
    plt.show()

component_chart(cur, "Top 15 skaters by WAR, 2025-26 (v0.4)", IMAGES / "war_top15_v04.png")

## 8. Save
- `war_v04.csv`: 2025-26, every skater. This feeds the first player cards.
- `war_by_season_v04.csv`: all five seasons, for working backwards.
- `war_history.csv`: v0.4 added to the version log.

In [ ]:
keep = ["season", "playerId", "name", "team", "position", "pos_group", "games_played",
        "all_toi", "ev_toi", "pp_toi", "pk_toi"] + COMPONENTS + ["gar", "off_war", "def_war", "other_war", "war", "war_rank"]
cur[keep].to_csv(DATA / "war_v04.csv", index=False)
v04[keep].to_csv(DATA / "war_by_season_v04.csv", index=False)

history = pd.read_csv(DATA / "war_history.csv")
new = cur[["playerId", "war", "war_rank"]].rename(columns={"war": "war_v04", "war_rank": "rank_v04"})
history = history.drop(columns=[c for c in new.columns if c != "playerId" and c in history.columns]).merge(new, on="playerId", how="left")
history.to_csv(DATA / "war_history.csv", index=False)
print("Saved", len(cur), "skaters (2025-26) and", len(v04), "player-seasons. war_history columns:", list(history.columns))

## Takeaways
- **v0.4 is a "what happened" WAR.** Goals above expected count fully. Team WAR matches team goal differential better than v0.3 in 3 of 5 seasons, about the same in the other 2, and better on average (0.79 vs 0.77, section 6).
- **Caufield is #1 for 2025-26.** That's what this definition says, and it's defensible: the goals happened. Project 2 will ask whether they repeat.
- **The defense stat stays as is.** Three alternatives were no meaningfully better. Its real limit (linemates and opponents) needs shift-level data.
- **Next:** the percentile table for the player cards, starting with 2025-26.